In [ ]:
import os
print(os.getcwd())

In [ ]:
import os
import ast
import pandas as pd
import osmnx as ox
import networkx as nx
import joblib
import route_network_analysis as rna

# Set working directory
#os.chdir("/proj/nobackup/streetnetwork-alignment/")
print("Current working directory:", os.getcwd())

def process_route(graph_files, row):

    print(f"Processing {row['city_name']} (ID: {row['id']})", flush=True)
    csv_path = os.path.join("experiment_routes", "csv_routes", f"{row['city_name']}_{row['id']}.csv")
    # Load graph
    filepath = graph_files.loc[graph_files["city_name"] == row["city_name"], "graph_file"].values[0]
    graph = ox.load_graphml(filepath)
    # Add weights and betweenness
    graph, _ = rna.street_network_analysis.add_deviation_from_prototypical_weights(graph)
    graph, _ = rna.street_network_analysis.add_instruction_equivalent_weights(graph)
    graph, _ = rna.street_network_analysis.add_node_degree_weights(graph)
    #ox.save_graphml(graph, filepath)
    # Process route
    old_complexity = row["sum_decision_complexity"]
    route_nodes = ast.literal_eval(row["nodes"])
    wstring = "length"
    if row["weight"] == "least_decision_complex":
        route_nodes = route_nodes[::-1]  # Reverse the list
        wstring = "decision_complexity"
    new_route_od_pair = rna.od_pair.from_route(graph, route_nodes, wstring)
    new_route_df = new_route_od_pair.get_odpair_df()
    # Add metadata
    new_route_df["old_complexity"] = old_complexity
    new_route_df["id"] = row["id"]
    new_route_df["complexity_difference"] = old_complexity - new_route_od_pair.path.complexity
    new_route_df["route_exp_condition"] = row["condition"]
    # Save to CSV
    new_route_df.to_csv(csv_path, index=False)
    print(f"Finished with route: {row['city_name']} (ID: {row['id']})", flush=True)
    return new_route_df


# Load data
route_data = pd.read_csv(os.path.join("experiment_routes", "route_data.csv"))
graph_files = pd.read_csv(os.path.join("experiment_routes", "graph_city_dicts.csv"))
graph_files["graph_file"] = graph_files["graph_file"].str.replace("\\", "/")

# Filter rows to only those that need processing
rows_to_process = [
    row for _, row in route_data.iterrows()
    if not os.path.exists(os.path.join("experiment_routes", "csv_routes", f"{row['city_name']}_{row['id']}.csv"))
]

# Process routes in parallel
num_processes = 4
odpair_dfs = joblib.Parallel(n_jobs=num_processes, backend='loky')(
    joblib.delayed(process_route)(graph_files, row) for row in rows_to_process
)
odpair_dfs = [df for df in odpair_dfs if df is not None]

# Concatenate and save
if odpair_dfs:
    df = pd.concat(odpair_dfs, ignore_index=True)
    df.to_csv(os.path.join("experiment_routes", "experiment_route_data.csv"), index=False)
    print("All routes processed and concatenated!", flush=True)
else:
    print("No routes were processed successfully.", flush=True)